In [ ]:
# 训练阶段
best_val = 0
global_step = 0
best_accuracy = 0
best_val_f1 = 0  # 记录最好的验证集F1

writer = SummaryWriter('logs')

# 循环训练
for epoch in range(1):
    model.train()
    train_loss = 0
    train_correct = 0      # 改为累计正确标签数
    train_total = 0         # 改为累计总标签数
    train_strict_correct = 0  # 累计严格正确样本数
    train_strict_total = 0    # 累计总样本数
    
    train_process = tqdm(train_loader, desc='Training')
    
    for image, labels, img_name in train_process:
        
        image = image.to(device)
        labels = labels.to(device)
    
        optimizer.zero_grad() # 清空之前的梯度
        outputs = model(image)  
        loss = criterion(outputs.logits, labels)  # outputs中包含了多个属性，需要转化成张量

        probability = torch.sigmoid(outputs.logits)
        predict = (probability > 0.5).float()

        
        # 1. 严格准确率
        strict_correct = (predict == labels).all(dim=1).sum().item()
        strict_total = labels.size(0)
        strict_acc = strict_correct / strict_total
        
        # 2. 宽松准确率
        total_correct = (predict == labels).sum().item()
        total_elements = labels.numel()  # batch_size * 8
        batch_acc = total_correct / total_elements
        
        loss.backward()  # 反向传播(计算梯度)
        optimizer.step() # 利用梯度来更新参数，优化

        # 累计损失
        train_loss += loss.item()
        
        # 累计两种准确率的计数
        train_strict_correct += strict_correct
        train_strict_total += strict_total
        
        train_correct += total_correct
        train_total += total_elements
        
        # 在进度条中显示宽松准确率
        train_process.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{batch_acc:.3f}',
            'strict': f'{strict_acc:.3f}'
        })

        writer.add_scalar('Batch/Train_Loss', loss.item(), global_step)
        writer.add_scalar('Batch/Train_Strict_Accuracy', strict_acc, global_step)
        writer.add_scalar('Batch/Train_Label_Accuracy', batch_acc, global_step)
        global_step += 1
    
    # ========== 计算epoch平均值 ==========
    # 严格准确率（用于保存最佳模型）
    train_strict_accuracy = train_strict_correct / train_strict_total
    # 宽松准确率（每个标签的准确率）
    train_label_accuracy = train_correct / train_total
    # 平均损失
    avg_train_loss = train_loss / len(train_loader)

    # 记录到TensorBoard
    writer.add_scalar('Loss/Train', avg_train_loss, epoch)
    writer.add_scalar('Accuracy/Train_Strict', train_strict_accuracy, epoch)
    writer.add_scalar('Accuracy/Train_Label', train_label_accuracy, epoch)
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Hyperparameters/Learning_Rate', current_lr, epoch)

    print(f"\nEpoch {epoch+1}:")
    print(f"  损失: {avg_train_loss:.4f}")
    print(f"  标签准确率: {train_label_accuracy:.3f} (每个标签)")
    print(f"  严格准确率: {train_strict_accuracy:.3f} (8标签全对)")

    # ========== 验证集评估（添加精确率、召回率、分类准确率） ==========
    model.eval()
    all_val_probs = []
    all_val_labels = []
    
    with torch.no_grad():
        for val_images, val_labels, _ in val_loader:
            val_images = val_images.to(device)
            val_outputs = model(val_images)
            val_probs = torch.sigmoid(val_outputs.logits).cpu().numpy()
            
            all_val_probs.append(val_probs)
            all_val_labels.append(val_labels.numpy())
    
    all_val_probs = np.vstack(all_val_probs)
    all_val_labels = np.vstack(all_val_labels)
    
    # 计算验证集预测结果
    val_preds = (all_val_probs > 0.5).astype(int)
    
    # ========== 新增：计算各种指标 ==========
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    
    # 1. 分类准确率（每个标签独立，等同于你的宽松准确率）
    val_accuracy = accuracy_score(all_val_labels.flatten(), val_preds.flatten())   # flatten():将多维展开成一维
    
    # 2. 宏平均精确率（每个类别独立计算后平均）
    val_precision = precision_score(all_val_labels, val_preds, average='macro', zero_division=0)
    
    # 3. 宏平均召回率
    val_recall = recall_score(all_val_labels, val_preds, average='macro', zero_division=0)
    
    # 4. 宏平均F1
    val_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)
    
    # 5. 各类别单独指标（用于分析少数类）
    per_class_precision = precision_score(all_val_labels, val_preds, average=None, zero_division=0)
    per_class_recall = recall_score(all_val_labels, val_preds, average=None, zero_division=0)
    per_class_f1 = f1_score(all_val_labels, val_preds, average=None, zero_division=0)
    
    print(f"\n📊 验证集评估结果:")
    print(f"  分类准确率 (Accuracy): {val_accuracy:.4f}")
    print(f"  宏平均精确率 (Precision): {val_precision:.4f}")
    print(f"  宏平均召回率 (Recall): {val_recall:.4f}")
    print(f"  宏平均F1 (Macro F1): {val_f1:.4f}")
    
    print(f"\n  各类别详细指标:")
    class_names = ['正常', 'DR', '青光眼', '白内障', '黄斑变性', '高血压', '近视', '其他']
    for i in range(8):
        print(f"    {class_names[i]}: 精确率={per_class_precision[i]:.3f}, 召回率={per_class_recall[i]:.3f}, F1={per_class_f1[i]:.3f}")
    
    # 计算少数类（类别4、5）的平均指标
    minority_precision = np.mean([per_class_precision[4], per_class_precision[5]])
    minority_recall = np.mean([per_class_recall[4], per_class_recall[5]])
    minority_f1 = np.mean([per_class_f1[4], per_class_f1[5]])
    print(f"\n  少数类(黄斑变性+高血压)平均: 精确率={minority_precision:.3f}, 召回率={minority_recall:.3f}, F1={minority_f1:.3f}")
    
    # 记录到TensorBoard（新增）
    writer.add_scalar('Val/Accuracy', val_accuracy, epoch)
    writer.add_scalar('Val/Precision', val_precision, epoch)
    writer.add_scalar('Val/Recall', val_recall, epoch)
    writer.add_scalar('Val/F1', val_f1, epoch)
    
    # 用验证集F1保存模型
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'strict_accuracy': train_strict_accuracy,
            'label_accuracy': train_label_accuracy,
            'val_f1': val_f1,
            'val_accuracy': val_accuracy,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'per_class_f1': per_class_f1,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model_by_val_f1.pth'))
        print(f"  ✅ 保存验证集最佳模型！F1={val_f1:.4f}")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_train_loss,
        'accuracy': train_strict_accuracy,
        'config': config
    }, os.path.join(config['save_dir'], 'latest_model_2.pth'))
    
    if train_strict_accuracy > best_accuracy:
        best_accuracy = train_strict_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'accuracy': train_strict_accuracy,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model_2.pth'))

writer.close()